# 🧬 CV DNA Pipeline - Google Colab Edition (FINAL - V2)
**Project**: Resume Ranking - Comprehensive CV Pre-processing

Notebook này triển khai Pipeline xử lý CV đa bước, từ bóc tách Header đến NER, Mapping Taxonomy và Phân tích thâm niên. Kết quả cuối cùng là hồ sơ năng lực chi tiết (DNA Profile) dạng JSON phục vụ chấm điểm.

In [ ]:

!pip install -q gliner chromadb sentence-transformers tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 916.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.4/170.4 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Kiểm tra cấu trúc ChromaDB kèm Embeddings
try:
    print("--- Danh sách Collections ---")
    collections = processor.chroma_client.list_collections()
    for col in collections:
        print(f"Collection: {col.name}")

    # Kiểm tra dữ liệu mẫu từ tech_ontology (bao gồm embeddings)
    print("\n--- Dữ liệu mẫu: tech_ontology (kèm Embeddings) ---")
    # Thêm include=['embeddings'] để hiển thị vector
    tech_sample = processor.tech_col.get(limit=2, include=['embeddings', 'metadatas', 'documents'])
    display(tech_sample)

    # Kiểm tra dữ liệu mẫu từ soft_skills_ontology
    print("\n--- Dữ liệu mẫu: soft_skills_ontology ---")
    soft_sample = processor.soft_col.get(limit=2, include=['embeddings', 'metadatas', 'documents'])
    display(soft_sample)
except Exception as e:
    print(f"Lỗi khi truy cập DB: {e}")

--- Danh sách Collections ---
Collection: tech_ontology
Collection: soft_skills_ontology

--- Dữ liệu mẫu: tech_ontology (kèm Embeddings) ---


{'ids': ['Software_Development.Frontend.Core_html',
  'Software_Development.Frontend.Core_html5'],
 'embeddings': array([[-2.18988676e-02,  3.73937450e-02, -5.87253608e-02,
         -8.10890924e-03,  3.08536235e-02, -3.29598226e-02,
         -2.24461928e-02, -3.12659070e-02,  7.68104792e-02,
         -7.50151649e-03,  3.69902179e-02,  1.00206278e-01,
          8.56853798e-02, -7.80573674e-03, -2.68800464e-02,
          2.58564949e-02,  1.16271591e-02,  2.45400257e-02,
          3.17451684e-03, -7.52568524e-03, -6.52011633e-02,
         -2.46863905e-02, -4.56542149e-02, -5.87470038e-03,
          2.57970430e-02,  6.09962046e-02,  2.33951549e-04,
          4.06129956e-02,  7.98125938e-02, -9.80476886e-02,
          3.79501693e-02, -1.16393931e-01,  9.72787663e-02,
          2.49296203e-02, -4.52690125e-02,  5.65539859e-02,
         -3.13159041e-02, -1.09415762e-02, -2.63270792e-02,
         -2.34154351e-02, -7.47516975e-02, -7.11947903e-02,
          2.10658535e-02,  2.33105775e-02,  4.6


--- Dữ liệu mẫu: soft_skills_ontology ---


{'ids': ['Attitudes, communication and social skills.attendance-management-case_commercial property',
  'Attitudes, communication and social skills.attendance-management-case_digital dictation case management system'],
 'embeddings': array([[ 5.32939881e-02,  2.97174957e-02, -2.52049998e-03,
          2.46617720e-02, -1.02999667e-02, -1.24082919e-02,
          4.71076146e-02, -1.50297005e-02, -3.48430946e-02,
         -4.09079716e-02,  4.36168611e-02, -4.82945107e-02,
          4.34020720e-03,  3.76617834e-02,  4.46654223e-02,
         -2.92903893e-02,  1.26797007e-02,  2.86193322e-02,
         -1.19927311e-02,  5.67659326e-02,  5.24344817e-02,
          4.36905473e-02,  3.69716785e-03, -5.93737839e-03,
          6.32469449e-03,  1.48055218e-02,  3.43343765e-02,
          9.10473466e-02,  6.82447925e-02, -5.02063930e-02,
          1.47304581e-02,  2.16791350e-02,  7.65012801e-02,
          1.44012980e-02,  1.00429013e-01,  6.33932948e-02,
         -8.79325159e-03, -2.97305360e-02,  3.1

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import json
import logging
from datetime import datetime
from tqdm.auto import tqdm
from gliner import GLiNER
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from dateutil import parser as date_parser

# Cấu hình Logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("/content/cv_processing.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

In [27]:
# === CONFIGURATION ===
BASE_PATH = "/content/drive/MyDrive/Nlp resume ranker"
INPUT_FILE = f"{BASE_PATH}/resumes_segment4/df_resumes_segmented_final.csv"
CHROMA_PATH = f"{BASE_PATH}/chroma_db"
OUTPUT_DIR = f"{BASE_PATH}/last_resumes"

# Tạo thư mục output nếu chưa có
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    logger.info(f"Created output directory: {OUTPUT_DIR}")

THRESHOLDS = {
    "technical": 0.25, # Distance (lower is better)
    "soft_skill": 0.25
}

In [42]:
class CVDNAProcessor:
    def __init__(self, chroma_path, model_ner="urchade/gliner_multi-v2.1", model_embed="all-MiniLM-L6-v2"):
        logger.info("Initializing models...")
        self.ner_model = GLiNER.from_pretrained(model_ner)
        self.embed_model = SentenceTransformer(model_embed)

        self.chroma_client = chromadb.PersistentClient(path=chroma_path)
        self.tech_col = self.chroma_client.get_collection("tech_ontology")
        self.soft_col = self.chroma_client.get_collection("soft_skills_ontology")
        logger.info("Models and DB loaded successfully.")

    def _safe_str(self, val):
        if val is None or val != val:
            return ""
        val_str = str(val).strip()
        if val_str.lower() in ['nan', 'none', 'null', 'n/a', '-', '']:
            return ""
        return val_str

    def parse_header(self, text):
        fields = ["Name", "Phone", "Email", "Location", "LinkedIn", "job_title", "years_of_experience"]
        info = {}
        for i, field in enumerate(fields):
            next_field_patterns = [f"{f}:" for f in fields[i+1:]]
            lookahead_pattern = '|'.join(next_field_patterns) if i < len(fields)-1 else '$'
            pattern = f"{field}:(.*?)(?={lookahead_pattern})"
            match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
            val = match.group(1).strip().rstrip(",;'\"") if match else None

            if val and val.lower() in ["none", "nan", "null", "n/a", "-", ""]:
                val = None

            if field == "years_of_experience":
                if val:
                    try:
                        match_num = re.search(r"\d+(\.\d+)?", val)
                        if match_num:
                            info[field] = round(float(match_num.group()), 1)
                        else:
                            info[field] = None
                    except:
                        info[field] = None
                else:
                    info[field] = None
            else:
                info[field] = val
        return info

    def chunk_text(self, text, size=200, stride=50):
        words = text.split()
        chunks = []
        for i in range(0, len(words), size - stride):
            chunk = " ".join(words[i:i + size])
            chunks.append(chunk)
            if i + size >= len(words): break
        return chunks

    def extract_entities(self, text, labels, zone_type, threshold=0.5):
        chunks = self.chunk_text(text)
        all_entities = []
        seen_uids = set()

        for chunk in chunks:
            entities = self.ner_model.predict_entities(chunk, labels, threshold=threshold)
            for ent in entities:
                text_val = ent['text'].strip()
                if (len(text_val) < 2 and text_val.upper() not in ["C", "R"]) or text_val.isdigit():
                    continue

                uid = f"{text_val.lower()}_{ent['label']}_{zone_type}"
                if uid not in seen_uids:
                    all_entities.append({"text": text_val, "label": ent['label'], "zone": zone_type})
                    seen_uids.add(uid)
        return all_entities

    def map_taxonomy(self, text, label, job_title_std=None, custom_threshold=None):
        is_soft = "Soft Skill" in label
        col = self.soft_col if is_soft else self.tech_col
        threshold = custom_threshold if custom_threshold is not None else (THRESHOLDS['soft_skill'] if is_soft else THRESHOLDS['technical'])

        results = col.query(query_texts=[text], n_results=3)
        if not results['ids'] or not results['ids'][0]:
            return None, None, label

        ids, dists, metadatas = results['ids'][0], results['distances'][0], results['metadatas'][0]
        if not ids or dists[0] > threshold:
            return None, None, label

        best_idx = 0
        if job_title_std:
            for i, meta in enumerate(metadatas):
                if job_title_std.lower() in str(meta.get('path', '')).lower():
                    best_idx = i
                    break

        mapped_id = ids[best_idx]
        mapped_path = metadatas[best_idx].get('path')
        final_label = label
        if not is_soft and "Certification" in label and mapped_path and "Tools" in mapped_path:
            final_label = "Technical Skill"

        return mapped_id, mapped_path, final_label

    def _merge_intervals(self, intervals):
        if not intervals:
            return []
        sorted_intervals = sorted(intervals, key=lambda x: x[0])
        merged = [sorted_intervals[0]]
        for current_start, current_end in sorted_intervals[1:]:
            last_start, last_end = merged[-1]
            if current_start <= last_end:
                merged[-1] = (last_start, max(last_end, current_end))
            else:
                merged.append((current_start, current_end))
        return merged

    def analyze_experience(self, exp_text, mapped_entities, limit_yoe=None):
        skill_intervals = {}
        date_pattern = r'(\b(?:[A-Za-z]+|\d{2}/)?\d{4})\s*[\u2013\u2014-]\s*(\b(?:[A-Za-z]+|\d{2}/)?\d{4}|Present|Now)'

        # Split blocks by 'Environment: [...]' ensuring each block ends after the environment array
        env_pattern = r'Environment:\s*\[.*?\]'
        matches = list(re.finditer(env_pattern, exp_text, re.IGNORECASE))
        blocks = []
        start_idx = 0
        for match in matches:
            end_idx = match.end()
            blocks.append(exp_text[start_idx:end_idx])
            start_idx = end_idx

        if start_idx < len(exp_text):
            remainder = exp_text[start_idx:].strip()
            if remainder:
                blocks.append(remainder)

        if not blocks:
            blocks = [exp_text]

        for block in blocks:
            date_match = re.search(date_pattern, block, re.IGNORECASE)
            if not date_match: continue
            try:
                start_str = date_match.group(1).replace("\u2013", "-").replace("\u2014", "-")
                end_str = date_match.group(2).replace("\u2013", "-").replace("\u2014", "-")
                start = date_parser.parse(start_str)
                end = datetime.now() if re.search(r'present|now', end_str, re.IGNORECASE) else date_parser.parse(end_str)
                if end < start:
                    continue
                block_lower = block.lower()
                for ent in mapped_entities:
                    skill_text = ent['text'].lower()

                    found = False
                    if skill_text in ["c++", "c#", "f#", ".net", "j#"]:
                        escaped = re.escape(skill_text)
                        found = bool(re.search(r'(?<![a-z0-9])' + escaped + r'(?![a-z0-9])', block_lower))
                    elif len(skill_text) <= 2:
                        escaped = re.escape(skill_text)
                        found = bool(re.search(r'\b' + escaped + r'\b', block_lower))
                    else:
                        found = skill_text in block_lower

                    if found:
                        key = ent['taxonomy_id'] if ent['taxonomy_id'] else skill_text
                        if key not in skill_intervals:
                            skill_intervals[key] = []
                        skill_intervals[key].append((start, end))
            except: continue

        skill_years = {}
        for key, intervals in skill_intervals.items():
            merged = self._merge_intervals(intervals)
            total_duration = sum((end - start).days / 365.25 for start, end in merged)
            skill_years[key] = max(0, total_duration)

        if limit_yoe is not None:
            for k in skill_years:
                if skill_years[k] > limit_yoe:
                    skill_years[k] = limit_yoe
        return skill_years

    def process_cv(self, row):
        try:
            header_text = self._safe_str(row.get('information_section'))
            header = self.parse_header(header_text)
            limit_yoe = header.get("years_of_experience")
            raw_entities = []

            # Removed 'Job Title' from labels list in all sections
            sections = [
                (self._safe_str(row.get('summary_objective')), "summary", ["Tool", "Methodology", "Technical Skill", "Soft Skill"], 0.4),
                (self._safe_str(row.get('technical_skills')), "skills_list", ["Technical Skill", "Tool", "Methodology", "Soft Skill"], 0.15),
                (self._safe_str(row.get('work_experience')), "experience", ["Technical Skill", "Tool", "Methodology", "Soft Skill"], 0.4),
                (self._safe_str(row.get('education_certifications')), "education", ["Education Major", "Certification"], 0.4)
            ]

            for text, zone, labels, thresh in sections:
                if text:
                    raw_entities += self.extract_entities(text, labels, zone, threshold=thresh)

            work_exp_text = self._safe_str(row.get('work_experience'))
            if work_exp_text:
                for env_match in re.finditer(r'Environment:\s*\[(.*?)\]', work_exp_text, re.IGNORECASE):
                    envs = [e.strip() for e in env_match.group(1).split(',') if e.strip()]
                    for e in envs:
                        if (len(e) >= 2 or e.upper() in ["C", "R"]) and not e.isdigit():
                            raw_entities.append({"text": e, "label": "Technical Skill", "zone": "environment"})

            job_title_std = None
            jt_val = header.get("job_title")
            if jt_val:
                 # Mapping Job Title still uses standard threshold for anchor
                 tid, _, _ = self.map_taxonomy(jt_val, "Job Title", custom_threshold=0.35)
                 job_title_std = tid

            mapped_results = []
            seen_map_uids = set()
            for ent in raw_entities:
                tid, path, label = self.map_taxonomy(ent['text'], ent['label'], job_title_std)
                muid = f"{ent['text'].lower()}_{label}_{ent['zone']}"
                if muid not in seen_map_uids:
                    mapped_results.append({"text": ent['text'], "taxonomy_id": tid, "path": path, "label": label, "zone": ent['zone']})
                    seen_map_uids.add(muid)

            skill_years = self.analyze_experience(work_exp_text, mapped_results, limit_yoe=limit_yoe) if work_exp_text else {}
            hard_skills, soft_skills, certs = {}, {}, set()
            education = {"Major": None, "Degree": None}
            majors_extracted = []

            for res in mapped_results:
                key = res['taxonomy_id'] if res['taxonomy_id'] else res['text'].lower()
                label = res['label']

                if "Soft Skill" in label:
                    if key not in soft_skills:
                        soft_skills[key] = {"skill": res['text'], "taxonomy_id": res['taxonomy_id']}
                elif "Certification" in label:
                    certs.add(res['text'])
                elif "Education Major" in label:
                    majors_extracted.append(res['text'])
                else:
                    if key not in hard_skills:
                        hard_skills[key] = {
                            "skill": res['text'],
                            "taxonomy_id": res['taxonomy_id'],
                            "years": round(skill_years.get(key, 0.0), 1),
                            "zones": {res['zone']}
                        }
                    else:
                        hard_skills[key]['zones'].add(res['zone'])
                        hard_skills[key]['years'] = round(skill_years.get(key, 0.0), 1)

            edu_text = self._safe_str(row.get('education_certifications'))
            edu_degree = None
            if re.search(r"(?i)PhD|Doctor|Tiến sĩ", edu_text): edu_degree = "PhD"
            elif re.search(r"(?i)Master|M\.S|M\.A|Thạc sĩ", edu_text): edu_degree = "Master"
            elif re.search(r"(?i)Bachelor|B\.S|B\.A|Cử nhân", edu_text): edu_degree = "Bachelor"

            education["Degree"] = edu_degree

            if majors_extracted:
                matched_major = None
                if edu_degree == "PhD": pattern = r"(?i)PhD|Doctor|Tiến sĩ"
                elif edu_degree == "Master": pattern = r"(?i)Master|M\.S|M\.A|Thạc sĩ"
                elif edu_degree == "Bachelor": pattern = r"(?i)Bachelor|B\.S|B\.A|Cử nhân"
                else: pattern = None

                if pattern:
                    for m in majors_extracted:
                        if re.search(pattern, m):
                            matched_major = m
                            break

                # Fallback: if no match found for degree inside major string, prefer the FIRST major found
                education["Major"] = matched_major if matched_major else majors_extracted[0]

            return {
                "Information": {
                    "Name": header.get("Name"), "Phone": header.get("Phone"), "Email": header.get("Email"),
                    "Location": header.get("Location"), "LinkedIn": header.get("LinkedIn"),
                    "Job_Title_Original": jt_val,
                    "Job_Title_Standardized": job_title_std,
                    "Years_of_Exp": limit_yoe
                },
                "Hard_Skills": {
                    "Direct_Mention": [ {**v, "zones": list(v['zones'])} for v in hard_skills.values() ],
                    "Certifications": list(certs)
                },
                "Soft_Skills": list(soft_skills.values()),
                "Education": education
            }
        except Exception as e:
            logger.error(f"Error processing CV: {e}")
            return None

## 🧪 Demo & Inspection
Chạy thử trên 1 CV đầu tiên để kiểm tra hiệu quả từng bước.

In [13]:
df = pd.read_csv(INPUT_FILE)
df.head()

,file_name,information_section,summary_objective,technical_skills,work_experience,education_certifications,new_word_count
0,Adelina_Erimia_PMP1.docx,"Name: Adelina Erimia, Phone: 469-331-7851, Ema...",Experienced project manager offering over 10 y...,"MS Project, PPM, Smartsheet, Microsoft Word, E...","[Project Management Institute (PMI), Savannah ...","Universitatea de Vest, Romania - BA - Business...",780
1,Jennifer M. Conte.docx,"Name: Jennifer M. Conte, job_title: Sr. Java D...",Jennifer M. Conte is a highly experienced Sr. ...,"Java/J2EE, JDK 1.8, Java 8, Java 6, C++, Angul...",[Macrosolid] | [Sr. JAVA Application Developer...,"Bachelor of Electronic Engineering, Beijing Un...",1124
2,Othman - Project Manager.docx,"Name: Othman, job_title: PROJECT MANAGER, year...",Intuitive IT management professional offering ...,"IT Management, Project Management, Organizatio...",[KO Satellite Inc.] | [IT Manager] | [2007 – P...,"Masters of Information System Management, Kell...",1196
3,Resume2018March.docx,"Name: Lorenzo Zackery, Phone: 215 . 980 . 1011...",I've been a WordPress Developer for 13 years n...,WordPress theme / plugin development & configu...,[Various WordPress Consulting Projects] | [Con...,NaN,1644
4,Randy Adams.docx,"Name: Randy Adams, Phone: [Missing], Email: ra...",Senior Java Developer with over 8 years of com...,"Languages: Core Java, Java (Generics), JavaScr...",[PG & E] | [Sr. Java Developer] | [04/2015 - P...,"Master of Science and Engineering, Brandeis Un...",844


In [14]:
df['education_certifications'][45]

'Bachelors Degree in Computer Science Engineering (2006 – 2010). Certifications: PMP, ITILV3, OCP.'

In [15]:
df['information_section'][45]

'Name: Candidate35, Phone: 518-805-9569, Email: candidate35@gmail.com, job_title: SR IT Project Manager and Oracle Database Administrator, years_of_experience: 10.0'

In [16]:
df['summary_objective'][45]

'Accomplished IT Project Manager and Oracle Database Administrator with over 10 years of experience across healthcare, retail, finance, and oil industries. Proven track record in managing large-scale infrastructure projects, including data center migrations to AWS, Citrix implementations, and server patching initiatives for over 1,500 assets. Expert in leading cross-functional onshore and offshore teams of up to 250 members while ensuring adherence to Agile, Waterfall, and Six Sigma methodologies. Highly skilled in technical business analysis, translating complex business needs into functional specifications, and maintaining rigorous quality assurance standards. Proficient in managing Access and Identity Management (AIM), GDPR compliance, and clinical data repository (CDR) reporting for state-level audits. Adept at leveraging tools like Clarity, MS Project, and ServiceNow to drive project milestones, mitigate risks, and optimize operational efficiency.'

In [17]:
df['technical_skills'][45]

'Project Management: Agile, Waterfall, Six Sigma, PMLC, SDLC/STLC, Vendor Management, Contract Management, Risk Management, Change Management, Incident/Problem Management, Budgeting, Resource Allocation. Databases: Oracle 10g/11g/12c, MS SQL Server 2000/2005/2008, RMAN, Data Guard, ASM, Oracle Database Vault, Flashback Technology. Operating Systems: AIX 5.2/5.3, Linux 3x/4x/5x, Sun Solaris 8/9, Windows NT/2000/2003. Tools & Utilities: Clarity, MS Project, ServiceNow, Remedy, ManageNow, Cherwell, OpsView, SCCM, Nexpose, Toad, GoldenGate, SQL Developer, TKPROF, AWR, ADDM, OEM, XML, SQL-Plus, SQL-Loader, BR Tools, Jira, Confluence, VMware. Languages: PL/SQL, SQL, C, SAP BR Tools. Compliance/Domains: GDPR, HIPAA, Health Insurance, Retail, Finance, Oil & Gas, OLTP, EDI, Clinical Data Repository (CDR).'

In [24]:
df['file_name'][20]

'Candidate_12.docx'

In [40]:
# Re-initialize processor with the updated class
processor = CVDNAProcessor(CHROMA_PATH)

test_row = df.iloc[20]
result = processor.process_cv(test_row)

# Hiển thị kết quả trích xuất để soi logic tính Year
import json
print(json.dumps(result, indent=2, ensure_ascii=False))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:04<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{
  "Information": {
    "Name": "Candidate12",
    "Phone": "681-888-2999",
    "Email": "Candidate12.java23@gmail.com",
    "Location": null,
    "LinkedIn": null,
    "Job_Title_Original": "Senior Java/J2EE Developer",
    "Job_Title_Standardized": null,
    "Years_of_Exp": 14.0
  },
  "Hard_Skills": {
    "Direct_Mention": [
      {
        "skill": "deep proficiency",
        "taxonomy_id": null,
        "years": 0.0,
        "zones": [
          "summary"
        ]
      },
      {
        "skill": "Docker",
        "taxonomy_id": "Infrastructure_and_DevOps.Containerization_Orchestration_docker",
        "years": 11.5,
        "zones": [
          "environment",
          "skills_list",
          "summary"
        ]
      },
      {
        "skill": "Mesos",
        "taxonomy_id": "Infrastructure_and_DevOps.Containerization_Orchestration_mesos",
        "years": 0.0,
        "zones": [
          "skills_list",
          "summary"
        ]
      },
      {
        "skill": "Stron

In [41]:
import pandas as pd

# 1. Lấy danh sách kỹ năng từ bản gốc (Technical Skills section)
raw_tech_skills = str(df.iloc[20]['technical_skills']).split(',')
raw_tech_skills = [s.strip() for s in raw_tech_skills]

# 2. Lấy danh sách kỹ năng đã trích xuất từ result
extracted_skills = result['Hard_Skills']['Direct_Mention']

# 3. Tạo bảng so sánh
comparison_data = []
for raw_s in raw_tech_skills:
    # Tìm xem kỹ năng gốc có nằm trong kết quả trích xuất không
    # Kiểm tra cả trường hợp kỹ năng gốc chứa kết quả trích xuất hoặc ngược lại
    match = next((s for s in extracted_skills if s['skill'].lower() in raw_s.lower() or raw_s.lower() in s['skill'].lower()), None)

    comparison_data.append({
        "Raw Skill from CV": raw_s,
        "Extracted": "✅" if match else "❌",
        "Mapped Taxonomy ID": match['taxonomy_id'] if match else "N/A"
    })

compare_df = pd.DataFrame(comparison_data)
display(compare_df)

# Thống kê nhanh
missing = compare_df[compare_df['Extracted'] == "❌"]
print(f"\n--- THỐNG KÊ ---")
print(f"- Tổng số kỹ năng liệt kê trong mục Tech Skills: {len(raw_tech_skills)}")
print(f"- Số kỹ năng GLiNER bỏ sót: {len(missing)}")
if not missing.empty:
    print(f"- Các kỹ năng bị sót: {missing['Raw Skill from CV'].tolist()}")

,Raw Skill from CV,Extracted,Mapped Taxonomy ID
0,Languages: Java/J2EE (JDK 1.6-1.8),✅,Software_Development.Backend.Java_Ecosystem_jdk
1,C,✅,None
2,C++,❌,N/A
3,COBOL,❌,N/A
4,JCL,✅,None
...,...,...,...
73,JMeter,✅,QA_and_Testing.Performance_Tools_jmeter
74,JProfiler,✅,None
75,SVN,✅,Infrastructure_and_DevOps.Version_Control_svn
76,GitHub,✅,Infrastructure_and_DevOps.Version_Control_github



--- THỐNG KÊ ---
- Tổng số kỹ năng liệt kê trong mục Tech Skills: 78
- Số kỹ năng GLiNER bỏ sót: 21
- Các kỹ năng bị sót: ['C++', 'COBOL', 'HTML5', 'CSS3', 'MVC', 'Security', 'Batch', 'AOP', 'DAO)', 'Hibernate 3.x/4.x/5.x', 'React JS', 'S3', 'VPC', 'RDS', 'Kubernetes', 'JAX-WS', 'JAX-RS', 'WSDL', 'UDDI', 'DB2', 'SQL Server']


## 🚀 Batch Processing
Xử lý toàn bộ tập dữ liệu và lưu vào Drive.

In [43]:
processed_count = 0
# Khởi tạo lại processor sau khi đã update class
processor = CVDNAProcessor(CHROMA_PATH)

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing CVs"):
    res = processor.process_cv(row)
    if res:
        # Sử dụng tên file gốc từ cột 'file_name' và loại bỏ đuôi .docx
        original_filename = str(row['file_name'])
        base_name = re.sub(r'\.(docx|doc|pdf)$', '', original_filename, flags=re.IGNORECASE)
        file_name = f"{base_name}.json"

        with open(os.path.join(OUTPUT_DIR, file_name), 'w', encoding='utf-8') as f:
            json.dump(res, f, indent=2, ensure_ascii=False)
        processed_count += 1

logger.info(f"Successfully processed {processed_count}/{len(df)} CVs.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Processing CVs:   0%|          | 0/228 [00:00<?, ?it/s]